In [1]:
import pandas as pd
from plotly.subplots import make_subplots
import plotly.graph_objects as go

df = pd.read_csv("prezzi_narcotraffico.csv")

# prezzo all'ingrosso riportato all'unità piccola (grammo o dose singola)
df["prezzo_ingrosso_unitario"] = df["prezzo_ingrosso"] / 1000
df["prezzo_dettaglio_medio"] = (df["prezzo_dettaglio_min"] + df["prezzo_dettaglio_max"]) / 2
df["incremento"] = df["prezzo_dettaglio_medio"] - df["prezzo_ingrosso_unitario"]

sostanze = df["sostanza"].tolist()

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=sostanze,
)

posizioni = [(1, 1), (1, 2), (2, 1), (2, 2)]

for riga_dati, (riga, colonna) in zip(df.itertuples(), posizioni):
    unita = "\u20ac/g" if riga_dati.unita_dettaglio == "eur_per_g" else "\u20ac/dose"
    fig.add_trace(
        go.Waterfall(
            orientation="v",
            measure=["absolute", "relative", "total"],
            x=["Ingrosso", "Incremento", "Dettaglio"],
            y=[riga_dati.prezzo_ingrosso_unitario, riga_dati.incremento, riga_dati.prezzo_dettaglio_medio],
            text=[f"{riga_dati.prezzo_ingrosso_unitario:.1f}", f"+{riga_dati.incremento:.1f}", f"{riga_dati.prezzo_dettaglio_medio:.1f}"],
            textposition="outside",
            textfont=dict(size=8),
            connector=dict(line=dict(color="#bbbbbb", width=1)),
            increasing=dict(marker=dict(color="#8c3b45")),
            totals=dict(marker=dict(color="#5d7ea8")),
            showlegend=False,
        ),
        row=riga, col=colonna,
    )
    fig.update_xaxes(title_text="Fase della filiera", row=riga, col=colonna, title_font=dict(size=10))
    fig.update_yaxes(title_text=f"Prezzo ({unita})", row=riga, col=colonna, title_font=dict(size=10))

fig.update_layout(
    template="simple_white",
    height=800,
    margin=dict(b=100),
)

# 3. NOTA FONTE (in basso)
fig.add_annotation(
    text=(""),
    xref="paper", yref="paper",
    x=0, y=-0.12,
    showarrow=False,
    font=dict(size=10, color="#888888"),
    align="left",
)

fig.show()